In [ ]:
import numpy as np
import pandas as pd

# Function to generate one batch of 10 points with fixed temperature
def generate_batch(num_points, temperature, bounds):
    batch = []

    for _ in range(num_points):
        point = [
            temperature,  # Fixed temperature
            #np.random.choice(FeCl2),  # Discrete pH
            #np.random.choice(MgCl2),  # Discrete rock type
            #np.random.choice(NaOH),  # Discrete grain size
            round_sig(np.random.uniform(bounds['FeCl2'][0], bounds['FeCl2'][1]), 2),
            round_sig(np.random.uniform(bounds['MgCl2'][0], bounds['MgCl2'][1]), 2),
            round_sig(np.random.uniform(bounds['NaOH'][0], bounds['NaOH'][1]), 2),
            round_sig(np.random.uniform(bounds['NaHCO3'][0], bounds['NaHCO3'][1]), 2),
            round_sig(np.random.uniform(bounds['NiCl2'][0], bounds['NiCl2'][1]), 2),
            round_sig(np.random.uniform(bounds['CuCl2'][0], bounds['CuCl2'][1]), 2),
            round_sig(np.random.uniform(bounds['NaCl'][0], bounds['NaCl'][1]), 2),
            round_sig(np.random.uniform(bounds['Surfactant'][0], bounds['Surfactant'][1]), 2)
        ]
        batch.append(point)

    return batch

# Helper function to round to 1 significant figure
def round_sig(x, sig=1):
    if x == 0:
        return 0
    return round(x, -int(np.floor(np.log10(abs(x)))) + (sig - 1))

# Parameter settings
temperatures = [25, 57.5, 90]
#ph_values = [4,5,6,7,8,9,10,11,12]
#rock_types = [0,1,2,3,4]
#grain_sizes = [45, 151, 375, 750, 1500]

# Bounds for continuous parameters
bounds = {
    'FeCl2': (0.05, 0.105),
    'MgCl2': (0.0, 0.805),
    'NaOH': (0.0, 0.825), # (0.0, 0.745) these are volume range
    'NaHCO3': (0.0, 5.005), # (0.0, 5.005)
    'NiCl2': (0.0, 0.0405), # (0.0, 0.4)
    'CuCl2': (0.0, 0.0405), # (0, 0.04)
    'NaCl': (0.0, 2.105), # (0, 2.13)
    'Surfactant': (0.0, 1.005) # (0, 0.4)
}

# Generate 3 batches of 10 points
all_points = []
for temp in temperatures:
    batch = generate_batch(10, temp, bounds)
    all_points.extend(batch)

# Create DataFrame
columns = [
    "Temperature", "FeCl2", "MgCl2", "NaOH", "NaHCO3", "NiCl2", "CuCl2", "NaCl", "Surfactant"
]
df = pd.DataFrame(all_points, columns=columns)

# Save to CSV
print(df)
df.to_csv("initialization_geo_hydrogen_HT.csv", index=False)
df.to_excel("initialization_geo_hydrogen_HT.xlsx", index=False)

from google.colab import drive
drive.mount('/content/drive')

save_path = '/content/drive/My Drive/Abate_Post_Doc/Bayesian_optimization_Tim/Addis_Ammonia/Initialization/initialization_synthetic_hydrogen_HT_2_Sig_figs2.xlsx'
df.to_excel(save_path, index=False)

In [ ]:
 # importing important libraries and Good setup. Ensures GPU usage and consistent numerical precision with torch.double.
import os
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.double
SMOKE_TEST = os.environ.get("SMOKE_TEST")

In [ ]:
!python --version

Python 3.11.13


In [ ]:
!pip install gpytorch==1.8.0 #initiall it was 1.8.0

In [ ]:
!pip install botorch==0.6.5

In [ ]:
import torch
import botorch

print(f"PyTorch version: {torch.__version__}")
print(f"BoTorch version: {botorch.__version__}")

PyTorch version: 2.6.0+cu124
BoTorch version: 0.6.5


In [ ]:
#importing libraries
import pandas as pd
from botorch.models import FixedNoiseGP, SingleTaskGP
from botorch.fit import fit_gpytorch_model
from botorch.acquisition.monte_carlo import qNoisyExpectedImprovement
from botorch.optim import optimize_acqf
from botorch.utils.transforms import normalize, unnormalize, standardize
from gpytorch.mlls import ExactMarginalLogLikelihood
from botorch.models.transforms.outcome import Standardize
from botorch.acquisition import qNoisyExpectedImprovement

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
file_path = '/content/drive/My Drive/Abate_Post_Doc/Bayesian_optimization_Tim/Addis_Ammonia/SHAP/Initialization_Data_SingleTask.xlsx'  # Replace with your actual file name

# Load the CSV file
data = pd.DataFrame(pd.read_excel(file_path))
#data = pd.read_csv(file_path)
# Display the first few rows to confirm
print(data.head())

print(data.columns.tolist())


In [ ]:
#Defining x and y data type
input_cols = ['Temperature', 'FeCl2', 'MgCl2', 'NaOH', 'NiCl2', 'CuCl2', 'NaHCO3', 'NaCl', 'Surfactant']
x = torch.tensor(data[input_cols].iloc[:33].to_numpy(), dtype=torch.double)
y_initial = data.iloc[:33, 10].values
y = torch.tensor(y_initial, dtype=torch.double).unsqueeze(-1)

In [ ]:
#Define bounds
bounds = torch.tensor([
    [25, 0.05, 0.0, 0.0, 0.0, 0.0, 0.0,0.0,0.0],  # Lower bounds for each parameter
    [90, 0.1, 0.8, 0.75, 0.4, 0.4, 5.0,2.13,0.4]  # Upper bounds for each parameter, it is because now I concentrated my stock solution, so actually 0.4 volume of surfactant will have same effect as 1.0 surfactant, and I normalize it before passing to BO
], dtype=torch.double)

In [ ]:
#Normalize X values
train_X_normalized = normalize(x, bounds)
train_X_normalized

In [ ]:
# Initialize the FixedNoiseGP model (work with botorch model 0.6.0)
model = SingleTaskGP(
    train_X=train_X_normalized,
    train_Y=y,
    outcome_transform=Standardize(m=y.shape[-1])  # Standardize the outputs
)

In [ ]:
# Initialize the model and marginal log-likelihood
mll = ExactMarginalLogLikelihood(model.likelihood, model)
mll.train()

# Fit the GP model
fit_gpytorch_model(mll)


In [ ]:
# Define the qNEI acquisition function
acqf = qNoisyExpectedImprovement(model=model, X_baseline=train_X_normalized)

#📦 Notes:
#Acquisition values should generally decrease over time as the model becomes confident and improvement plateaus.

#If acquisition value stays high but yield doesn't improve, the model may be exploring too much or getting stuck — worth debugging or using a different acquisition function (e.g., UCB).

#You can also log posterior mean, variance, or even model uncertainty if needed.

In [ ]:
# Define normalized bounds [0, 1]
bounds_normalized = torch.tensor([[0.0] * bounds.size(1), [1.0] * bounds.size(1)], dtype=torch.double)
#bounds_normalized=torch.tensor([[0.0] * x_normalized.size(1), [1.0] * x_normalized.size(1)], dtype=torch.double)

In [ ]:
# Initialize round counter
round_number = 1

In [ ]:
torch.manual_seed(52)

acquisition_values = [] #should decrease over time for better performance and if it remains high, the optimizer is stuck, try different acquisition function

# Perform 10 iterations of Bayesian optimization
for i in range(1):
    # Optimize the acquisition function to find the next candidate in normalized space
    candidate_normalized, _ = optimize_acqf(
        acq_function=acqf,
        bounds=bounds_normalized,
        q=15,  # Number of candidates to select
        num_restarts=50,  # Number of restarts for optimization
        raw_samples=256, # Number of random samples to start with
    )
    final_candidate_normalized = candidate_normalized
    round_number += 1

    acqf_val = acqf(candidate_normalized).item()
    acquisition_values.append(acqf_val)


# Unnormalize the final candidate to the original space
final_candidate = unnormalize(final_candidate_normalized, bounds)
final_candidate[:, 1] = torch.round(final_candidate[:, 1] * 1000) / 1000 #rounding off is done here #FeCL2
final_candidate[:, 2] = torch.round(final_candidate[:, 2] * 1000) / 1000 #MgCl2
final_candidate[:, 3] = torch.round(final_candidate[:, 3]* 1000) / 1000 #rounding off is done here #NaOH
final_candidate[:, 4] = torch.round(final_candidate[:, 4]* 1000) / 1000 #rounding off is done here #NiCl2
final_candidate[:, 5] = torch.round(final_candidate[:, 5] * 1000) / 1000 #CuCl2
final_candidate[:, 6] = torch.round(final_candidate[:, 6] * 100) / 100 # NaHCO3
final_candidate[:, 7] = torch.round(final_candidate[:, 7] * 1000) / 1000 #NaCl
final_candidate[:, 8] = torch.round(final_candidate[:, 8] * 1000) / 1000 #Surfactant

# Modify the second parameter (index 1)
# 1. Extract the second parameter column
second_param = final_candidate[:, 0]

# 2. Get top 5 highest values
top5_values, _ = torch.topk(second_param, 5)

# 3. Compute their average
average_top5 = top5_values.mean()

# 4. Replace all second parameter values with the average
final_candidate[:, 0] = average_top5

print("Final candidate in original space after 10 iterations:", final_candidate)
#candidate_unnormalized[0] = torch.round(candidate_unnormalized[0])  # Discretize integer-constrained parameter for rounding

if isinstance(final_candidate, torch.Tensor):
    final_candidate = final_candidate.numpy()

# Ensure it's a 2D array (reshape if necessary)
final_candidate = final_candidate.reshape(-1, 9)

In [ ]:
data1 = data.rename(columns=lambda x: x.strip())
data1

In [ ]:
# 1. Convert new candidates to DataFrame (only 9 columns for now)
new_candidates_df = pd.DataFrame(final_candidate, columns=[
    'Temperature', 'FeCl2', 'MgCl2', 'NaOH', 'NiCl2', 'CuCl2', 'NaHCO3', 'NaCl', 'Surfactant'
])

# 2. Insert a blank column for y without creating an extra column
if new_candidates_df.shape[1] == data1.shape[1] - 1:
    # Add a blank column at the end
    new_candidates_df[data1.columns[-1]] = None  # Use the same name as existing last column (e.g., "y")

# 3. Concatenate with existing data
df_updated = pd.concat([data1, new_candidates_df], ignore_index=True)

# 4. Save to new Excel file
save_path = f'Round_{round_number}.xlsx'
df_updated.to_excel(save_path, index=False)

# Also save to your output path
output_path = '/content/drive/My Drive/Abate_Post_Doc/Bayesian_optimization_Tim/Addis_Ammonia/SingleTask_Round_2_Synthetic_Rock_15batch1.xlsx'
df_updated.to_excel(output_path, index=False)

# 5. Update for next batch
df_existing = df_updated.copy()
round_number += 1

In [ ]:
new_data = pd.DataFrame(torch.tensor(final_candidate).numpy(), columns=data.columns[:9])
new_data
